# GenZify: Millennial → Gen Z Language Transformer
**NLP Lab Assignment: Sequence-to-Sequence Encoder–Decoder (LSTM) with Attention**

This notebook implements every part of the assignment end to end:

| Assignment part | Where in this notebook |
|---|---|
| Dataset upload, 80/10/10 split, quality and leakage checks | §1–2 |
| Task 1: Preprocessing (normalisation, tokenisation, vocab, `<SOS>/<EOS>/<PAD>/<UNK>`, padding) + written answer | §3 |
| Task 2: LSTM Encoder (embeddings → LSTM → hidden & cell states) | §4 |
| Task 3: LSTM Decoder (starts at `<SOS>`, stops at `<EOS>`/max length) | §4 |
| Task 5: Attention (Model A fixed context vs Model B Bahdanau attention) | §4, §8, §10 |
| Task 4: Training (teacher forcing, Adam, cross-entropy), hyperparameter experiments, loss plots | §5, §7, §8 |
| Task 6: Greedy search and beam search | §6, §10 (Exp. B) |
| Evaluation: BLEU, semantic similarity, train/val loss | §9 |
| Experiments A–E | §10 |
| Human evaluation (3 evaluators) | §11 |
| Bonus 1–5 (tone selector, multiple outputs, reverse model, context-aware slang, unknown slang) | §12 |
| GenZify app (Gradio) | §13 |
| Saved models, plots, results, auto-generated comparison report | §14 |

**How to run:** `Runtime → Change runtime type → GPU (T4)`, then `Runtime → Run all`.
The cell in §1 opens a file picker: upload **`genzify_parallel_dataset_2000.csv`**.
Full run on a T4 GPU takes roughly 10–15 minutes (set `RUN_HPARAM_SEARCH = False` to skip the
hyperparameter sweep and save time). The human-evaluation cell is the only one that waits on you.

## 0. Setup

In [ ]:
!pip install -q sacrebleu sentence-transformers gradio

In [ ]:
import io, os, re, math, json, time, random, copy, zipfile, unicodedata
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

import sacrebleu
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed()

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

OUT_DIR = 'genzify_outputs'
os.makedirs(OUT_DIR, exist_ok=True)

# Global switches
RUN_HPARAM_SEARCH = True     # Task 4 sweep (~5-8 min on T4). Set False to skip.
SEARCH_EPOCHS     = 25       # epochs per sweep run (early stopping still applies)
FINAL_EPOCHS      = 60       # max epochs for the final models (early stopping applies)
PATIENCE          = 8

## 1. Upload the dataset
The dataset is **not** expected to be in the Colab file system: it is uploaded through
`google.colab.files.upload()`. Pick `genzify_parallel_dataset_2000.csv` in the dialog.

In [ ]:
from google.colab import files

uploaded = files.upload()                      # opens the file picker
DATA_FILE = next(iter(uploaded))               # name of the uploaded file
print('Uploaded:', DATA_FILE, f'({len(uploaded[DATA_FILE]):,} bytes)')

df = pd.read_csv(io.BytesIO(uploaded[DATA_FILE]), encoding='utf-8-sig')
df.columns = [c.strip().lstrip('﻿') for c in df.columns]
df.head()

## 2. Dataset exploration and quality checks
The assignment's dataset rules: ≥500 pairs (recommended 1,000–2,000), 80/10/10 split, meaningful
targets, variety of lengths/sentence types, varied slang, and **no duplicate or near-duplicate
pairs across train and test**. We check each of these.

In [ ]:
REQUIRED = ['source_text', 'target_text', 'split']
assert all(c in df.columns for c in REQUIRED), f'CSV must contain {REQUIRED}'
df = df.dropna(subset=['source_text', 'target_text']).reset_index(drop=True)
df['split'] = df['split'].str.strip().str.lower().replace({'val': 'validation', 'valid': 'validation', 'dev': 'validation'})

print('Total pairs:', len(df))
print('\nSplit sizes:\n', df['split'].value_counts())
print('\nSplit proportions:\n', (df['split'].value_counts(normalize=True) * 100).round(1))
if 'category' in df.columns:
    print('\nCategories:\n', df['category'].value_counts())
if 'tone' in df.columns:
    print('\nTones:\n', df['tone'].value_counts())

df['src_words'] = df['source_text'].str.split().str.len()
df['trg_words'] = df['target_text'].str.split().str.len()
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].hist(df['src_words'], bins=range(0, 20), color='#4C72B0', edgecolor='white'); ax[0].set_title('Source length (words)')
ax[1].hist(df['trg_words'], bins=range(0, 20), color='#DD8452', edgecolor='white'); ax[1].set_title('Target length (words)')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/length_distribution.png', dpi=120); plt.show()

In [ ]:
# Sentence-type coverage (rule 4) and slang variety (rule 5)
df['is_question'] = df['source_text'].str.strip().str.endswith('?')
print('Questions in source:', df['is_question'].sum(), '| statements:', (~df['is_question']).sum())

SLANG_LEXICON = ['no cap', 'fr', 'rn', 'lowkey', 'highkey', 'vibe', 'vibing', 'fire', 'cooked', 'slay', 'bussin',
                 'locked in', 'cringe', 'mid', 'w', 'l', 'ate', 'bet', 'sus', 'hyped', 'dead', 'goated', 'era',
                 'main character', 'in my feels', 'the move', 'big w', 'crew', 'ain\'t no way', 'it\'s giving',
                 'serving', 'unhinged', 'deadass', 'bestie', 'periodt', 'iykyk', 'rizz', 'touch grass', 'valid']
def slang_terms(text):
    t = ' ' + re.sub(r"[^a-z' ]", ' ', text.lower()) + ' '
    return [s for s in SLANG_LEXICON if f' {s} ' in t]
slang_counts = Counter(s for t in df['target_text'] for s in slang_terms(t))
print('\nMost frequent slang in targets:', slang_counts.most_common(15))
print('Distinct slang expressions used:', len(slang_counts))

In [ ]:
# Duplicate / near-duplicate leakage check (rule 6).
# Many sentences are a "base" sentence plus a context phrase ("... at the cafe."). If the same base
# sentence appeared in train AND test, test scores would be inflated. We strip the context phrase
# and check whether any base sentence crosses splits.
CONTEXT_PHRASES = ['right now', 'before class', 'after lunch', 'this weekend', 'at the campus event',
                   'during the trip', 'after a long day', 'before the deadline', 'during the holidays',
                   'at the cafe', 'on the way home', 'during the break', 'after the meeting',
                   'before the concert', 'at the library', 'during the presentation', 'after the exam',
                   'while travelling', 'at the weekend meetup', 'with my friends']
_ctx_pat = re.compile(r'\s+(?:' + '|'.join(map(re.escape, CONTEXT_PHRASES)) + r')\s*([.?!]?)$', re.I)
df['base'] = df['source_text'].apply(lambda s: _ctx_pat.sub(r'\1', s.strip()).lower())

exact_dupes = df.duplicated(subset=['source_text', 'target_text']).sum()
bases_per_split = df.groupby('base')['split'].nunique()
crossing = bases_per_split[bases_per_split > 1]
train_bases = set(df.loc[df.split == 'train', 'base'])
test_leak = df.loc[df.split == 'test', 'base'].isin(train_bases).mean()
print(f'Exact duplicate pairs: {exact_dupes}')
print(f'Distinct base sentences: {df.base.nunique()}')
print(f'Base sentences appearing in more than one split: {len(crossing)}')
print(f'Share of test pairs whose base sentence is in train: {test_leak:.1%}')

if len(crossing) > 0:
    # Re-split by base-sentence group so no near-duplicate crosses splits
    print('Leakage found -> re-splitting by base-sentence groups (80/10/10).')
    groups = df['base'].unique().tolist(); random.Random(SEED).shuffle(groups)
    n = len(groups); g_tr = set(groups[:int(.8 * n)]); g_va = set(groups[int(.8 * n):int(.9 * n)])
    df['split'] = df['base'].apply(lambda b: 'train' if b in g_tr else ('validation' if b in g_va else 'test'))
else:
    print('No near-duplicate leakage: the provided split is already group-aware, so we keep it.')

train_df = df[df.split == 'train'].reset_index(drop=True)
val_df   = df[df.split == 'validation'].reset_index(drop=True)
test_df  = df[df.split == 'test'].reset_index(drop=True)
print(f'\ntrain={len(train_df)}  validation={len(val_df)}  test={len(test_df)}')

Because each test sentence's *base* never appears in training, the test set is a genuine
generalisation test: the model must transform sentences whose core content it has never seen.
Scores are therefore lower than they would be with a random row-level split, but they are honest.

## 3. Task 1: Text preprocessing
Steps: **normalise** (Unicode NFKC, lower-case, unify quotes, collapse whitespace) → **tokenise**
(word-level regex that keeps contractions like `i'm` and `that's` as single tokens and splits
punctuation/emoji into their own tokens) → **build vocabularies** from the training split only →
**numericalise** (token → ID) → add `<SOS>`/`<EOS>` → **pad** each batch with `<PAD>`.
Words never seen in training map to `<UNK>`.

In [ ]:
PAD, SOS, EOS, UNK = '<PAD>', '<SOS>', '<EOS>', '<UNK>'
SPECIALS = [PAD, SOS, EOS, UNK]
PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3

TOKEN_RE = re.compile(r"[a-z0-9]+(?:'[a-z]+)?|[^\w\s]", re.UNICODE)

def normalize(text: str) -> str:
    text = unicodedata.normalize('NFKC', str(text))
    text = text.replace('’', "'").replace('‘', "'").replace('“', '"').replace('”', '"')
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)
    return text

def tokenize(text: str):
    return TOKEN_RE.findall(normalize(text))

def detokenize(tokens):
    s = ' '.join(tokens)
    s = re.sub(r'\s+([.,!?;:])', r'\1', s)          # no space before punctuation
    s = re.sub(r'\bi\b', 'I', s)                     # i -> I
    s = re.sub(r"\bi'", "I'", s)                     # i'm -> I'm
    return s[:1].upper() + s[1:] if s else s

class Vocab:
    def __init__(self, token_lists, min_freq=1):
        self.freq = Counter(t for toks in token_lists for t in toks)
        self.itos = SPECIALS + sorted(t for t, c in self.freq.items() if c >= min_freq and t not in SPECIALS)
        self.stoi = {t: i for i, t in enumerate(self.itos)}
    def __len__(self): return len(self.itos)
    def encode(self, tokens, add_sos_eos=True):
        ids = [self.stoi.get(t, UNK_IDX) for t in tokens]
        return [SOS_IDX] + ids + [EOS_IDX] if add_sos_eos else ids
    def decode(self, ids):
        out = []
        for i in ids:
            if i == EOS_IDX: break
            if i in (SOS_IDX, PAD_IDX): continue
            out.append(self.itos[i])
        return out

def build_vocabs(train_frame, src_col='source_text', trg_col='target_text', min_freq=1):
    return (Vocab([tokenize(s) for s in train_frame[src_col]], min_freq),
            Vocab([tokenize(t) for t in train_frame[trg_col]], min_freq))

SRC_VOCAB, TRG_VOCAB = build_vocabs(train_df)
print('Source vocab size:', len(SRC_VOCAB), '| Target vocab size:', len(TRG_VOCAB))

ex = train_df.iloc[0]
print('\nExample source :', ex.source_text)
print('Tokens         :', tokenize(ex.source_text))
print('IDs            :', SRC_VOCAB.encode(tokenize(ex.source_text)))
print('Example target :', ex.target_text)
print('IDs            :', TRG_VOCAB.encode(tokenize(ex.target_text)))

def unk_rate(frame, vocab, col):
    toks = [t for s in frame[col] for t in tokenize(s)]
    return sum(t not in vocab.stoi for t in toks) / max(1, len(toks))
print(f'\nOOV (<UNK>) rate  val source: {unk_rate(val_df, SRC_VOCAB, "source_text"):.2%} | '
      f'test source: {unk_rate(test_df, SRC_VOCAB, "source_text"):.2%} | '
      f'test target: {unk_rate(test_df, TRG_VOCAB, "target_text"):.2%}')

In [ ]:
FUNCTION_WORDS = set("""a an the i me my we our you your he she it they them this that these those is am are was were be
been being to of in on at for with by from up about into over after before during while and or but so not no do does
did have has had will would can could should very really just too what why how when where who which there here all
any some more most much many such than then now right today tonight i'm that's it's let's""".split())

class ParallelDataset(Dataset):
    """Stores token lists; numericalises on the fly.

    word_dropout (training only): a content word that appears in BOTH the source and the target is
    replaced by <UNK> on both sides with probability p. The model thereby learns "an unknown word in
    the input usually reappears at the aligned place in the output", which lets us copy unseen words
    (names, objects) through at inference time via attention-based <UNK> replacement (§6)."""
    def __init__(self, frame, src_vocab, trg_vocab, src_col='source_text', trg_col='target_text', word_dropout=0.0):
        self.src = [tokenize(s) for s in frame[src_col]]
        self.trg = [tokenize(t) for t in frame[trg_col]]
        self.sv, self.tv, self.p = src_vocab, trg_vocab, word_dropout
    def __len__(self): return len(self.src)
    def __getitem__(self, i):
        s, t = self.src[i], self.trg[i]
        if self.p > 0:
            shared = {w for w in set(s) & set(t) if w.isalpha() and w not in FUNCTION_WORDS}
            drop = {w for w in shared if random.random() < self.p}
            if drop:
                s = [UNK if w in drop else w for w in s]; t = [UNK if w in drop else w for w in t]
        return torch.tensor(self.sv.encode(s)), torch.tensor(self.tv.encode(t))

def collate(batch):
    """Pad every sequence in the batch to the batch's max length with <PAD> (id 0)."""
    srcs, trgs = zip(*batch)
    src_lens = torch.tensor([len(s) for s in srcs])
    src = nn.utils.rnn.pad_sequence(srcs, batch_first=True, padding_value=PAD_IDX)
    trg = nn.utils.rnn.pad_sequence(trgs, batch_first=True, padding_value=PAD_IDX)
    return src, src_lens, trg

def make_loaders(batch_size, src_vocab=None, trg_vocab=None, src_col='source_text', trg_col='target_text', word_dropout=0.1):
    sv, tv = src_vocab or SRC_VOCAB, trg_vocab or TRG_VOCAB
    g = torch.Generator(); g.manual_seed(SEED)
    tr = DataLoader(ParallelDataset(train_df, sv, tv, src_col, trg_col, word_dropout), batch_size, shuffle=True,
                    collate_fn=collate, generator=g)
    va = DataLoader(ParallelDataset(val_df, sv, tv, src_col, trg_col), batch_size, shuffle=False, collate_fn=collate)
    return tr, va

src_b, len_b, trg_b = next(iter(make_loaders(4)[0]))
print('Padded source batch (ids):\n', src_b)
print('Lengths:', len_b.tolist())

**Q: Why do encoder–decoder models require numerical token representations instead of raw text?**

Neural networks only compute with numbers: every layer is matrix multiplication plus non-linearities.
Raw strings have no arithmetic meaning, so text must first be mapped to integers (token IDs) that
index rows of an **embedding matrix**, which turns each ID into a dense, trainable vector where
similar words can end up close together. IDs also give us:
* a **fixed, finite output space**: the decoder's softmax is a probability distribution over
  vocabulary IDs, and cross-entropy loss needs a target class index at every step;
* **batching**: sequences of different lengths become equal-length integer tensors via `<PAD>`,
  which the loss and attention can mask out;
* **control tokens**: `<SOS>` tells the decoder to start, `<EOS>` tells it when to stop, and
  `<UNK>` gives every unseen word a defined representation.

## 4. Tasks 2, 3 and 5: Encoder, Decoders and Attention

**Encoder (Task 2):** token IDs → embedding → (bi-directional) multi-layer LSTM. It returns
(a) the output at every time step (used by attention) and (b) the final **hidden state h** and
**cell state c** of every layer. With a bi-directional LSTM the forward and backward final states
are concatenated and projected (a "bridge" layer with tanh) to the decoder's size.

**Model A: basic decoder (Task 3):** initialised with the encoder's final (h, c), which act as the
single **fixed context vector** for the whole sentence. At each step it embeds the previous token,
runs one LSTM step and predicts the next token through a linear layer + softmax.

**Model B: attention decoder (Task 5):** Bahdanau (additive) attention. At every decoding step

$$e_{t,i} = v^\top \tanh(W_d\,s_{t-1} + W_e\,h_i),\quad \alpha_{t,i} = \mathrm{softmax}_i(e_{t,i}),\quad c_t = \sum_i \alpha_{t,i} h_i$$

so the decoder builds a *different* context vector $c_t$ for each output token by looking back at
all encoder states $h_i$. Padding positions are masked out. The LSTM input is `[embedding; c_t]`
and the output layer sees `[s_t; c_t; embedding]`.

Both models share the same encoder and the same `step()` interface, so training, greedy decoding
and beam search work for both unchanged.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers, dropout, bidirectional=True):
        super().__init__()
        self.n_layers, self.hid_dim, self.bidirectional = n_layers, hid_dim, bidirectional
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, batch_first=True, bidirectional=bidirectional,
                           dropout=dropout if n_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        d = 2 if bidirectional else 1
        self.out_dim = hid_dim * d
        # bridge: map [forward; backward] final states -> decoder hidden size
        self.bridge_h = nn.Linear(hid_dim * d, hid_dim)
        self.bridge_c = nn.Linear(hid_dim * d, hid_dim)

    def forward(self, src, src_lens):
        emb = self.dropout(self.embedding(src))                           # (B, S, E)
        packed = pack_padded_sequence(emb, src_lens.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, (h, c) = self.rnn(packed)
        outputs, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=src.size(1))  # (B, S, H*d)
        if self.bidirectional:
            # h: (n_layers*2, B, H) -> (n_layers, B, 2H)
            h = h.view(self.n_layers, 2, h.size(1), self.hid_dim).permute(0, 2, 1, 3).reshape(self.n_layers, h.size(1), -1)
            c = c.view(self.n_layers, 2, c.size(1), self.hid_dim).permute(0, 2, 1, 3).reshape(self.n_layers, c.size(1), -1)
        h = torch.tanh(self.bridge_h(h)); c = torch.tanh(self.bridge_c(c))  # (n_layers, B, H)
        return outputs, h.contiguous(), c.contiguous()


class BasicDecoder(nn.Module):
    """Model A: fixed context (only the encoder's final h, c)."""
    uses_attention = False
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers, dropout, enc_out_dim=None):
        super().__init__()
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, batch_first=True, dropout=dropout if n_layers > 1 else 0.0)
        self.fc_out = nn.Linear(hid_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def step(self, tok, h, c, enc_outputs=None, mask=None):
        emb = self.dropout(self.embedding(tok.unsqueeze(1)))             # (B, 1, E)
        out, (h, c) = self.rnn(emb, (h, c))
        logits = self.fc_out(out.squeeze(1))                             # (B, V)
        return logits, h, c, None


class BahdanauAttention(nn.Module):
    def __init__(self, enc_dim, dec_dim, attn_dim):
        super().__init__()
        self.W_e = nn.Linear(enc_dim, attn_dim, bias=False)
        self.W_d = nn.Linear(dec_dim, attn_dim, bias=False)
        self.v = nn.Linear(attn_dim, 1, bias=False)

    def forward(self, dec_hidden, enc_outputs, mask, enc_proj=None):
        # dec_hidden: (B, H) top-layer state; enc_outputs: (B, S, Henc); mask: (B, S) True=real token
        if enc_proj is None:
            enc_proj = self.W_e(enc_outputs)
        scores = self.v(torch.tanh(enc_proj + self.W_d(dec_hidden).unsqueeze(1))).squeeze(-1)  # (B, S)
        scores = scores.masked_fill(~mask, -1e9)
        alpha = F.softmax(scores, dim=-1)
        context = torch.bmm(alpha.unsqueeze(1), enc_outputs).squeeze(1)  # (B, Henc)
        return context, alpha


class AttentionDecoder(nn.Module):
    """Model B: a new context vector c_t at every step via Bahdanau attention."""
    uses_attention = True
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers, dropout, enc_out_dim):
        super().__init__()
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.attention = BahdanauAttention(enc_out_dim, hid_dim, hid_dim)
        self.rnn = nn.LSTM(emb_dim + enc_out_dim, hid_dim, n_layers, batch_first=True,
                           dropout=dropout if n_layers > 1 else 0.0)
        self.fc_out = nn.Linear(hid_dim + enc_out_dim + emb_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def step(self, tok, h, c, enc_outputs, mask, enc_proj=None):
        emb = self.dropout(self.embedding(tok))                          # (B, E)
        context, alpha = self.attention(h[-1], enc_outputs, mask, enc_proj)
        out, (h, c) = self.rnn(torch.cat([emb, context], -1).unsqueeze(1), (h, c))
        logits = self.fc_out(torch.cat([out.squeeze(1), context, emb], -1))
        return logits, h, c, alpha


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder, self.decoder = encoder, decoder

    def encode(self, src, src_lens):
        enc_out, h, c = self.encoder(src, src_lens)
        mask = src != PAD_IDX
        enc_proj = self.decoder.attention.W_e(enc_out) if self.decoder.uses_attention else None
        return enc_out, h, c, mask, enc_proj

    def dec_step(self, tok, h, c, enc_out, mask, enc_proj):
        if self.decoder.uses_attention:
            return self.decoder.step(tok, h, c, enc_out, mask, enc_proj)
        return self.decoder.step(tok, h, c)

    def forward(self, src, src_lens, trg, teacher_forcing_ratio=0.5):
        B, T = trg.shape
        enc_out, h, c, mask, enc_proj = self.encode(src, src_lens)
        logits_all = torch.zeros(B, T, self.decoder.vocab_size, device=src.device)
        tok = trg[:, 0]                                                   # <SOS>
        for t in range(1, T):
            logits, h, c, _ = self.dec_step(tok, h, c, enc_out, mask, enc_proj)
            logits_all[:, t] = logits
            use_tf = random.random() < teacher_forcing_ratio            # teacher forcing
            tok = trg[:, t] if use_tf else logits.argmax(-1)
        return logits_all


def build_model(cfg, attention, src_vocab=None, trg_vocab=None):
    sv, tv = src_vocab or SRC_VOCAB, trg_vocab or TRG_VOCAB
    enc = Encoder(len(sv), cfg['emb_dim'], cfg['hid_dim'], cfg['n_layers'], cfg['dropout'], cfg.get('bidirectional', True))
    Dec = AttentionDecoder if attention else BasicDecoder
    dec = Dec(len(tv), cfg['emb_dim'], cfg['hid_dim'], cfg['n_layers'], cfg['dropout'], enc.out_dim)
    model = Seq2Seq(enc, dec).to(DEVICE)
    for name, p in model.named_parameters():                              # sensible init
        if 'weight' in name and p.dim() > 1 and 'embedding' not in name:
            nn.init.xavier_uniform_(p)
        elif 'bias' in name:
            nn.init.zeros_(p)
    return model

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

_tmp_cfg = dict(emb_dim=256, hid_dim=512, n_layers=2, dropout=0.3)
print('Model A params:', f'{count_params(build_model(_tmp_cfg, attention=False)):,}')
print('Model B params:', f'{count_params(build_model(_tmp_cfg, attention=True)):,}')
print(build_model(_tmp_cfg, attention=True))

## 5. Task 4: Training loop
* **Loss:** `CrossEntropyLoss(ignore_index=<PAD>)` over every target position after `<SOS>`
  (label smoothing 0.1 to soften over-confidence on a small dataset).
* **Optimiser:** Adam, gradient-norm clipping at 1.0, `ReduceLROnPlateau` on validation loss.
* **Teacher forcing:** with probability `tf_ratio` the ground-truth previous token is fed to the
  decoder instead of its own prediction.
* **Validation loss** is the teacher-forced cross-entropy on the validation set (same quantity as
  the training loss, so the two curves are directly comparable).
* **Model selection / early stopping** uses **validation BLEU with greedy decoding**, i.e. the
  model is judged the way it is actually used. On a dataset this small the validation loss starts
  rising (over-confidence) several epochs before generation quality stops improving, so stopping on
  loss alone would keep an under-trained model. Both curves are plotted.
* **Word dropout** (p = 0.1, see §3) regularises the model and teaches it to copy unknown words.

In [ ]:
def make_criterion():
    return nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.1)

def run_epoch(model, loader, criterion, optimizer=None, tf_ratio=0.5, clip=1.0):
    train = optimizer is not None
    model.train(train)
    total, n_tok = 0.0, 0
    with torch.set_grad_enabled(train):
        for src, src_lens, trg in loader:
            src, trg = src.to(DEVICE), trg.to(DEVICE)
            logits = model(src, src_lens, trg, tf_ratio if train else 0.0)
            V = logits.size(-1)
            loss = criterion(logits[:, 1:].reshape(-1, V), trg[:, 1:].reshape(-1))
            if train:
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), clip); optimizer.step()
            k = (trg[:, 1:] != PAD_IDX).sum().item()
            total += loss.item() * k; n_tok += k
    return total / n_tok

def train_model(cfg, attention, epochs, patience=PATIENCE, name='model', verbose=True,
                loaders=None, src_vocab=None, trg_vocab=None, val_pairs=None):
    set_seed()
    model = build_model(cfg, attention, src_vocab, trg_vocab)
    tr_loader, va_loader = loaders or make_loaders(cfg['batch_size'])
    vs, vr = val_pairs or (val_df['source_text'].tolist(), val_df['target_text'].tolist())
    criterion = make_criterion()
    opt = torch.optim.Adam(model.parameters(), lr=cfg['lr'])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=3)
    hist = {'train_loss': [], 'val_loss': [], 'val_bleu': []}
    best, best_state, bad = -1.0, None, 0
    t0 = time.time()
    for ep in range(1, epochs + 1):
        tr = run_epoch(model, tr_loader, criterion, opt, cfg['tf_ratio'])
        va = run_epoch(model, va_loader, criterion, tf_ratio=1.0)
        bleu = corpus_bleu(generate(model, vs, src_vocab=src_vocab, trg_vocab=trg_vocab), vr)
        sched.step(bleu)
        hist['train_loss'].append(tr); hist['val_loss'].append(va); hist['val_bleu'].append(bleu)
        if bleu > best + 1e-3:
            best, best_state, bad = bleu, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
        if verbose and (ep == 1 or ep % 5 == 0):
            print(f'[{name}] epoch {ep:3d} | train loss {tr:.4f} | val loss {va:.4f} | val BLEU {bleu:5.2f} '
                  f'| lr {opt.param_groups[0]["lr"]:.1e} | {time.time()-t0:.0f}s')
        if bad >= patience:
            if verbose: print(f'[{name}] early stopping at epoch {ep}')
            break
    model.load_state_dict(best_state)
    hist['best_epoch'] = int(np.argmax(hist['val_bleu']) + 1)
    hist['best_val_bleu'] = best
    hist['best_val'] = hist['val_loss'][hist['best_epoch'] - 1]     # val loss at the selected epoch
    if verbose: print(f'[{name}] selected epoch {hist["best_epoch"]} | val BLEU {best:.2f} | val loss {hist["best_val"]:.4f}')
    return model, hist

def plot_history(hists, title, fname):
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    for (label, h), col in zip(hists.items(), ['#4C72B0', '#DD8452', '#55A868', '#C44E52']):
        ep = range(1, len(h['train_loss']) + 1)
        ax[0].plot(ep, h['train_loss'], '-', color=col, label=f'{label} train')
        ax[0].plot(ep, h['val_loss'], '--', color=col, label=f'{label} validation')
        ax[1].plot(ep, h['val_bleu'], '-', color=col, label=label)
        ax[1].axvline(h['best_epoch'], color=col, ls=':', alpha=.6)
    ax[0].set(xlabel='epoch', ylabel='cross-entropy loss', title='Training vs validation loss')
    ax[1].set(xlabel='epoch', ylabel='BLEU', title='Validation BLEU (dotted = selected epoch)')
    for a in ax: a.legend(fontsize=8); a.grid(alpha=.3)
    fig.suptitle(title); plt.tight_layout(); plt.savefig(f'{OUT_DIR}/{fname}', dpi=120); plt.show()

## 6. Decoding helpers (needed for validation BLEU during the sweep)
### Task 6A: Greedy search
At every step take the arg-max token, feed it back, stop at `<EOS>` or the maximum length.
Batched for speed.

**Repetition blocking.** Small seq2seq models trained on little data tend to loop ("catchy catchy
catchy"). Both decoders forbid any token that would repeat a bigram already generated
(`NO_REPEAT_NGRAM = 2`); set it to 0 to see the raw model behaviour.

**Unknown-word handling.** About a fifth of the words in test sentences never occur in training
(e.g. "jacket", "assignments"), so they become `<UNK>`. Thanks to word dropout the attention model
learns to emit `<UNK>` where such a word belongs in the output; `ids_to_tokens` then replaces it with
the source word that received the highest attention weight at that step (Luong et al., 2015). This
is what lets the model keep content words it has never seen. Model A has no attention, so it can
only drop them, which is part of why it preserves meaning worse.

In [ ]:
def max_len_for(n_src_tokens):
    return min(2 * n_src_tokens + 10, 80)

def ids_to_tokens(ids, tv, attn=None, src_tokens=None):
    """Convert output ids to tokens. <UNK> replacement (Luong et al., 2015): when the attention model
    emits <UNK>, copy the source word it attends to most at that step. Without attention, <UNK> is dropped."""
    out = []
    for t, i in enumerate(ids):
        if i == EOS_IDX: break
        if i in (SOS_IDX, PAD_IDX): continue
        if i == UNK_IDX:
            if attn is not None and src_tokens:
                row = np.asarray(attn[t])[1:len(src_tokens) + 1]          # skip <SOS>/<EOS> positions
                out.append(src_tokens[int(row.argmax())])
            continue
        out.append(tv.itos[i])
    return out

NO_REPEAT_NGRAM = 2   # set 0 to disable repetition blocking

def banned_next_tokens(history, n=NO_REPEAT_NGRAM):
    """Tokens that would repeat an n-gram already in `history` (no-repeat-n-gram blocking).
    n=2 also forbids immediate duplicates like "is is". <UNK> is never banned."""
    banned = set()
    if n >= 1 and len(history) >= n - 1:
        if n == 1: return set(history) - {UNK_IDX}
        prefix = tuple(history[len(history) - (n - 1):])
        for i in range(len(history) - n + 1):
            if tuple(history[i:i + n - 1]) == prefix:
                banned.add(history[i + n - 1])
    if history: banned.add(history[-1])
    banned.discard(UNK_IDX)
    return banned

@torch.no_grad()
def greedy_decode_batch(model, sentences, src_vocab=None, trg_vocab=None, return_attn=False):
    sv, tv = src_vocab or SRC_VOCAB, trg_vocab or TRG_VOCAB
    model.eval()
    encoded = [torch.tensor(sv.encode(tokenize(s))) for s in sentences]
    lens = torch.tensor([len(e) for e in encoded])
    src = nn.utils.rnn.pad_sequence(encoded, batch_first=True, padding_value=PAD_IDX).to(DEVICE)
    enc_out, h, c, mask, enc_proj = model.encode(src, lens)
    B, L = len(sentences), max_len_for(int(lens.max()))
    tok = torch.full((B,), SOS_IDX, dtype=torch.long, device=DEVICE)
    out_ids, attns = [], []
    for _ in range(L):
        logits, h, c, alpha = model.dec_step(tok, h, c, enc_out, mask, enc_proj)
        if NO_REPEAT_NGRAM and out_ids:
            hist_ids = torch.stack(out_ids, 1).tolist()
            for b in range(B):
                ban = banned_next_tokens(hist_ids[b])
                if ban: logits[b, list(ban)] = -1e9
        tok = logits.argmax(-1)
        out_ids.append(tok); attns.append(alpha)
    out_ids = torch.stack(out_ids, 1).tolist()
    A = torch.stack(attns, 1).cpu().numpy() if model.decoder.uses_attention else None   # (B, T, S)
    src_toks = [tokenize(s) for s in sentences]
    results = [ids_to_tokens(ids, tv, A[i] if A is not None else None, src_toks[i]) for i, ids in enumerate(out_ids)]
    if return_attn and A is not None:
        n_out = [next((t for t, x in enumerate(ids) if x == EOS_IDX), len(ids)) for ids in out_ids]
        return results, [A[i, :n_out[i] + 1, :lens[i]] for i in range(B)]
    return results

def generate(model, sentences, method='greedy', beam_width=5, **kw):
    if method == 'greedy':
        out = []
        for i in range(0, len(sentences), 128):
            out += greedy_decode_batch(model, sentences[i:i + 128], **kw)
        return [detokenize(t) for t in out]
    return [detokenize(beam_search(model, s, beam_width, **kw)[0][0]) for s in sentences]

def corpus_bleu(hyps, refs):
    """Corpus BLEU (sacreBLEU, 13a tokenisation, lower-cased)."""
    return sacrebleu.corpus_bleu([h.lower() for h in hyps], [[r.lower() for r in refs]]).score

### Task 6B: Beam search
Keep the `beam_width` most probable partial sequences. Each step expands every live hypothesis
with its top-k next tokens, scores them by summed log-probability, keeps the best `k`, and moves
hypotheses that emit `<EOS>` to a finished list. Final ranking uses **length normalisation**
(score / length^α, α = 0.7) so the search does not simply favour short outputs. It also returns
the n-best list, which Bonus 2 uses for multiple alternatives.

In [ ]:
@torch.no_grad()
def beam_search(model, sentence, beam_width=5, alpha=0.7, n_best=1, src_vocab=None, trg_vocab=None,
                return_attn=False):
    sv, tv = src_vocab or SRC_VOCAB, trg_vocab or TRG_VOCAB
    model.eval()
    src_tokens = tokenize(sentence)
    ids = sv.encode(src_tokens)
    src = torch.tensor([ids], device=DEVICE); lens = torch.tensor([len(ids)])
    enc_out, h, c, mask, enc_proj = model.encode(src, lens)
    L = max_len_for(len(ids))
    # live beams: (token list, logprob, h, c, attn list)
    beams = [([SOS_IDX], 0.0, h, c, [])]
    finished = []
    for _ in range(L):
        k = len(beams)
        tok = torch.tensor([b[0][-1] for b in beams], device=DEVICE)
        H = torch.cat([b[2] for b in beams], 1); C = torch.cat([b[3] for b in beams], 1)
        e_out = enc_out.expand(k, -1, -1); m = mask.expand(k, -1)
        e_proj = enc_proj.expand(k, -1, -1) if enc_proj is not None else None
        logits, H, C, att = model.dec_step(tok, H, C, e_out, m, e_proj)
        logp = F.log_softmax(logits, -1)                                  # (k, V)
        if NO_REPEAT_NGRAM:
            for b in range(k):
                ban = banned_next_tokens(beams[b][0][1:])
                if ban: logp[b, list(ban)] = -1e9
        cand = (logp + torch.tensor([b[1] for b in beams], device=DEVICE).unsqueeze(1)).view(-1)
        top_lp, top_idx = cand.topk(min(2 * beam_width, cand.numel()))
        new_beams = []
        for lp, idx in zip(top_lp.tolist(), top_idx.tolist()):
            b, t = divmod(idx, logp.size(1))
            seq = beams[b][0] + [t]
            att_hist = beams[b][4] + ([att[b].cpu().numpy()] if att is not None else [])
            if t == EOS_IDX:
                finished.append((seq, lp / (len(seq) - 1) ** alpha, att_hist))
            else:
                new_beams.append((seq, lp, H[:, b:b + 1], C[:, b:b + 1], att_hist))
            if len(new_beams) == beam_width: break
        beams = new_beams
        if not beams or len(finished) >= max(beam_width, n_best) and \
           max(f[1] for f in finished) > max(b[1] / (len(b[0])) ** alpha for b in beams):
            break
    if len(finished) < n_best:   # fall back to unfinished beams
        finished += [(b[0], b[1] / len(b[0]) ** alpha, b[4]) for b in beams]
    finished.sort(key=lambda x: -x[1])
    outs, seen = [], set()
    for seq, score, att_hist in finished:
        toks = ids_to_tokens(seq[1:], tv, att_hist if att_hist else None, src_tokens)
        key = ' '.join(toks)
        if key in seen: continue
        seen.add(key)
        outs.append((toks, score, np.array(att_hist) if att_hist else None))
        if len(outs) == n_best: break
    if return_attn: return outs
    return [(t, s) for t, s, _ in outs]

## 7. Task 4: Hyperparameter experiments
One-factor-at-a-time sweep around a baseline (attention model), varying embedding dimension,
hidden size, number of LSTM layers, learning rate, batch size and teacher-forcing ratio. The
number of epochs is studied through the curves plus early stopping (the selected epoch is reported).
Selection criterion: **validation BLEU** (greedy), with validation loss reported alongside.

In [ ]:
BASE_CFG = dict(emb_dim=128, hid_dim=256, n_layers=2, dropout=0.3, lr=1e-3, batch_size=32, tf_ratio=0.5, bidirectional=True)

SWEEP = [('baseline', {})] + [
    ('emb_dim=64', dict(emb_dim=64)), ('emb_dim=256', dict(emb_dim=256)),
    ('hid_dim=128', dict(hid_dim=128)), ('hid_dim=512', dict(hid_dim=512)),
    ('n_layers=1', dict(n_layers=1)), ('n_layers=3', dict(n_layers=3)),
    ('lr=3e-4', dict(lr=3e-4)), ('lr=3e-3', dict(lr=3e-3)),
    ('batch=16', dict(batch_size=16)), ('batch=64', dict(batch_size=64)),
    ('tf=0.25', dict(tf_ratio=0.25)), ('tf=0.75', dict(tf_ratio=0.75)), ('tf=1.0', dict(tf_ratio=1.0)),
]

val_src, val_ref = val_df['source_text'].tolist(), val_df['target_text'].tolist()
sweep_rows, sweep_hists = [], {}
if RUN_HPARAM_SEARCH:
    for name, change in SWEEP:
        cfg = {**BASE_CFG, **change}
        t0 = time.time()
        m, h = train_model(cfg, attention=True, epochs=SEARCH_EPOCHS, patience=6, name=name, verbose=False)
        bleu = corpus_bleu(generate(m, val_src), val_ref)
        sweep_rows.append(dict(run=name, **{k: cfg[k] for k in ['emb_dim', 'hid_dim', 'n_layers', 'lr', 'batch_size', 'tf_ratio']},
                               val_loss_at_best=round(h['best_val'], 4), best_epoch=h['best_epoch'],
                               val_BLEU=round(bleu, 2), params=count_params(m), secs=round(time.time() - t0)))
        sweep_hists[name] = h
        print(f"{name:12s} val_BLEU={bleu:5.2f} (epoch {h['best_epoch']:2d})  val_loss={h['best_val']:.4f}  {time.time()-t0:.0f}s")
        del m; torch.cuda.empty_cache() if torch.cuda.is_available() else None
    sweep_df = pd.DataFrame(sweep_rows).sort_values('val_BLEU', ascending=False)
    sweep_df.to_csv(f'{OUT_DIR}/hyperparameter_sweep.csv', index=False)
    display(sweep_df)
    best_run = sweep_df.iloc[0]['run']
    FINAL_CFG = {**BASE_CFG, **dict(SWEEP)[best_run]}
    print('Selected configuration:', best_run, FINAL_CFG)
else:
    sweep_df = None
    FINAL_CFG = dict(BASE_CFG)
    print('Sweep skipped; using baseline configuration:', FINAL_CFG)

In [ ]:
if RUN_HPARAM_SEARCH:
    fig, axes = plt.subplots(2, 3, figsize=(15, 7.5), sharey=True)
    groups = {'Embedding dim': ['emb_dim=64', 'baseline', 'emb_dim=256'], 'Hidden size': ['hid_dim=128', 'baseline', 'hid_dim=512'],
              'LSTM layers': ['n_layers=1', 'baseline', 'n_layers=3'], 'Learning rate': ['lr=3e-4', 'baseline', 'lr=3e-3'],
              'Batch size': ['batch=16', 'baseline', 'batch=64'], 'Teacher forcing': ['tf=0.25', 'baseline', 'tf=0.75', 'tf=1.0']}
    base_label = {'Embedding dim': 'emb=128', 'Hidden size': 'hid=256', 'LSTM layers': 'layers=2', 'Learning rate': 'lr=1e-3',
                  'Batch size': 'batch=32', 'Teacher forcing': 'tf=0.5'}
    for ax, (title, runs) in zip(axes.ravel(), groups.items()):
        for r in [r for r in runs if r in sweep_hists]:
            ax.plot(range(1, len(sweep_hists[r]['val_bleu']) + 1), sweep_hists[r]['val_bleu'],
                    label=r.replace('baseline', f'baseline ({base_label[title]})'))
        ax.set_title(title); ax.set_xlabel('epoch'); ax.set_ylabel('validation BLEU (greedy)'); ax.legend(fontsize=8); ax.grid(alpha=.3)
    plt.suptitle('Task 4: validation BLEU under each hyperparameter change'); plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/hyperparameter_sweep_curves.png', dpi=120); plt.show()

## 8. Train the final Model A (basic) and Model B (attention)
Both use the identical selected configuration; the only difference is the decoder.

In [ ]:
model_A, hist_A = train_model(FINAL_CFG, attention=False, epochs=FINAL_EPOCHS, name='Model A (basic)')
model_B, hist_B = train_model(FINAL_CFG, attention=True,  epochs=FINAL_EPOCHS, name='Model B (attention)')
plot_history({'Model A (basic)': hist_A}, 'Model A: training vs validation loss', 'loss_model_A.png')
plot_history({'Model B (attention)': hist_B}, 'Model B: training vs validation loss', 'loss_model_B.png')
plot_history({'Model A (basic)': hist_A, 'Model B (attention)': hist_B}, 'Model A vs Model B loss curves', 'loss_A_vs_B.png')
print(f"Selected epoch / val BLEU / val loss   A: {hist_A['best_epoch']} / {hist_A['best_val_bleu']:.2f} / {hist_A['best_val']:.4f}"
      f"   |   B: {hist_B['best_epoch']} / {hist_B['best_val_bleu']:.2f} / {hist_B['best_val']:.4f}")

In [ ]:
# Quick sanity check on the illustrative examples from the assignment
DEMO = ['I am extremely tired today.', 'This is really impressive.', 'I cannot believe this happened.',
        'I am very excited about this.', 'This situation is embarrassing.', 'I completely agree with you.',
        'I am not interested in this.', 'That was an amazing experience.',
        'I have an important exam tomorrow and I have not prepared anything.',
        'I really enjoyed spending time with my friends.']
pd.DataFrame({'input': DEMO,
              'Model A (greedy)': generate(model_A, DEMO),
              'Model B (greedy)': generate(model_B, DEMO),
              'Model B (beam=5)': generate(model_B, DEMO, 'beam')})

## 9. Evaluation metrics
* **BLEU** (sacreBLEU corpus BLEU; plus smoothed sentence BLEU for per-sentence analysis):
  n-gram overlap with the single reference. With one reference and many valid paraphrases,
  BLEU under-rewards correct but differently-worded slang.
* **Semantic similarity:** cosine similarity of `all-MiniLM-L6-v2` sentence embeddings, computed
  two ways: **prediction ↔ reference** (does it say what the reference says?) and
  **prediction ↔ source** (meaning preservation, the assignment's core requirement; the
  reference ↔ source similarity is reported as the ceiling a human-written target reaches).
* **Loss curves:** see §8. **Human evaluation:** see §11.

In [ ]:
from sentence_transformers import SentenceTransformer
sim_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=str(DEVICE))

def sem_sim(a_list, b_list):
    ea = sim_model.encode(a_list, convert_to_tensor=True, normalize_embeddings=True, batch_size=128)
    eb = sim_model.encode(b_list, convert_to_tensor=True, normalize_embeddings=True, batch_size=128)
    return (ea * eb).sum(-1).cpu().numpy()

_smooth = SmoothingFunction().method4
def sent_bleu(h, r):
    return 100 * sentence_bleu([tokenize(r)], tokenize(h), smoothing_function=_smooth)

def evaluate_outputs(srcs, hyps, refs=None):
    res = {'sim(pred, source)': float(np.mean(sem_sim(hyps, srcs)))}
    if refs is not None:
        res['BLEU'] = corpus_bleu(hyps, refs)
        res['sim(pred, reference)'] = float(np.mean(sem_sim(hyps, refs)))
    res['avg length (words)'] = float(np.mean([len(h.split()) for h in hyps]))
    res['exact match %'] = float(np.mean([h.lower().strip(' .') == r.lower().strip(' .') for h, r in zip(hyps, refs)]) * 100) if refs else np.nan
    return res

test_src, test_ref = test_df['source_text'].tolist(), test_df['target_text'].tolist()
t0 = time.time()
PRED = {
    ('A', 'greedy'): generate(model_A, test_src, 'greedy'),
    ('A', 'beam'):   generate(model_A, test_src, 'beam', beam_width=5),
    ('B', 'greedy'): generate(model_B, test_src, 'greedy'),
    ('B', 'beam'):   generate(model_B, test_src, 'beam', beam_width=5),
}
print(f'Decoding done in {time.time()-t0:.0f}s')

rows = []
for (m, dec), hyps in PRED.items():
    rows.append({'model': {'A': 'A: basic', 'B': 'B: attention'}[m], 'decoding': dec, **evaluate_outputs(test_src, hyps, test_ref)})
ref_row = {'model': 'Human reference', 'decoding': '-', 'sim(pred, source)': float(np.mean(sem_sim(test_ref, test_src)))}
results_df = pd.DataFrame(rows + [ref_row]).round(3)
results_df.to_csv(f'{OUT_DIR}/test_results.csv', index=False)
print('TEST SET RESULTS (unseen base sentences)')
display(results_df)

## 10. Experiments A–E

### Experiment A: Basic vs Attention model
Compare on the test set (table above), per category, and visualise what attention learns.

In [ ]:
cmp = test_df[['source_text', 'target_text']].copy()
cmp['A_greedy'] = PRED[('A', 'greedy')]; cmp['B_greedy'] = PRED[('B', 'greedy')]
cmp['bleu_A'] = [sent_bleu(h, r) for h, r in zip(cmp.A_greedy, cmp.target_text)]
cmp['bleu_B'] = [sent_bleu(h, r) for h, r in zip(cmp.B_greedy, cmp.target_text)]
cmp['simsrc_A'] = sem_sim(cmp.A_greedy.tolist(), test_src); cmp['simsrc_B'] = sem_sim(cmp.B_greedy.tolist(), test_src)
if 'category' in test_df.columns:
    cmp['category'] = test_df['category']
    by_cat = cmp.groupby('category')[['bleu_A', 'bleu_B', 'simsrc_A', 'simsrc_B']].mean().round(2)
    display(by_cat)
    by_cat[['bleu_A', 'bleu_B']].plot.barh(figsize=(8, 5), color=['#4C72B0', '#DD8452'])
    plt.xlabel('mean sentence BLEU'); plt.title('Experiment A: BLEU by category'); plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/expA_bleu_by_category.png', dpi=120); plt.show()
print(f"Attention wins on {(cmp.bleu_B > cmp.bleu_A).mean():.0%} of test sentences, ties {(cmp.bleu_B == cmp.bleu_A).mean():.0%}")
display(cmp[['source_text', 'target_text', 'A_greedy', 'B_greedy']].sample(12, random_state=1))

In [ ]:
def plot_attention(sentence, model=None, fname=None):
    model = model or model_B
    (toks,), (A,) = greedy_decode_batch(model, [sentence], return_attn=True)
    src_toks = [SOS] + tokenize(sentence) + [EOS]
    fig, ax = plt.subplots(figsize=(0.55 * len(src_toks) + 2, 0.45 * (len(toks) + 1) + 1.5))
    ax.imshow(A, cmap='viridis', aspect='auto')
    ax.set_xticks(range(len(src_toks))); ax.set_xticklabels(src_toks, rotation=60, ha='right')
    ax.set_yticks(range(len(toks) + 1)); ax.set_yticklabels(toks + [EOS])
    ax.set_title(f'Attention: "{detokenize(toks)}"', fontsize=10)
    plt.tight_layout()
    if fname: plt.savefig(f'{OUT_DIR}/{fname}', dpi=120)
    plt.show()

for i, s in enumerate([test_src[0], test_src[57], 'I am feeling exhausted after the exam.']):
    plot_attention(s, fname=f'attention_example_{i}.png')

### Experiment B: Greedy vs Beam search

In [ ]:
beam_rows = []
for k in [1, 3, 5, 10]:
    hyps = generate(model_B, test_src, 'beam', beam_width=k) if k > 1 else PRED[('B', 'greedy')]
    r = evaluate_outputs(test_src, hyps, test_ref)
    beam_rows.append({'decoding': 'greedy' if k == 1 else f'beam k={k}', 'BLEU': r['BLEU'],
                      'sim(pred, reference)': r['sim(pred, reference)'], 'sim(pred, source)': r['sim(pred, source)'],
                      'avg length': r['avg length (words)']})
beam_df = pd.DataFrame(beam_rows).round(3); display(beam_df)
beam_df.to_csv(f'{OUT_DIR}/expB_greedy_vs_beam.csv', index=False)

gb = pd.DataFrame({'source': test_src, 'reference': test_ref, 'greedy': PRED[('B', 'greedy')], 'beam k=5': PRED[('B', 'beam')]})
diff = gb[gb.greedy != gb['beam k=5']]
print(f'Beam search changed the output for {len(diff)}/{len(gb)} test sentences. Examples:')
display(diff.head(12))

### Experiment C: Short vs Long sentences
The dataset's sentences are 5–15 words, so the 20–30-word bucket is built by **joining two
unseen test pairs** with ", and" (source and reference joined the same way). This keeps a
reference for BLEU and directly tests whether a single fixed context vector loses information
as inputs grow longer than anything seen in training.

In [ ]:
def n_words(s): return len(s.split())
short_df = test_df[test_df.source_text.apply(n_words).between(5, 8)]
med_df = test_df[test_df.source_text.apply(n_words).between(10, 15)]

rng = random.Random(SEED); long_pairs = []
idx = list(range(len(test_df)))
while len(long_pairs) < 120:
    i, j = rng.sample(idx, 2)
    s = test_df.source_text[i].rstrip('.?!') + ', and ' + test_df.source_text[j][0].lower() + test_df.source_text[j][1:]
    t = test_df.target_text[i].rstrip('.?!') + ', and ' + test_df.target_text[j][0].lower() + test_df.target_text[j][1:]
    if 20 <= n_words(s) <= 30: long_pairs.append((s, t))
long_df = pd.DataFrame(long_pairs, columns=['source_text', 'target_text'])

len_rows = []
for bucket, frame in [('short (5-8)', short_df), ('medium (10-15)', med_df), ('long (20-30)', long_df)]:
    s, r = frame.source_text.tolist(), frame.target_text.tolist()
    for m_name, m in [('A: basic', model_A), ('B: attention', model_B)]:
        hyps = generate(m, s)
        ev = evaluate_outputs(s, hyps, r)
        len_rows.append({'bucket': bucket, 'n': len(s), 'model': m_name, 'BLEU': ev['BLEU'],
                         'sim(pred, reference)': ev['sim(pred, reference)'], 'sim(pred, source)': ev['sim(pred, source)'],
                         'output/reference length': np.mean([n_words(h) for h in hyps]) / np.mean([n_words(x) for x in r])})
len_df = pd.DataFrame(len_rows).round(3); display(len_df)
len_df.to_csv(f'{OUT_DIR}/expC_length.csv', index=False)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for m_name, col in [('A: basic', '#4C72B0'), ('B: attention', '#DD8452')]:
    sub = len_df[len_df.model == m_name]
    ax[0].plot(sub.bucket, sub.BLEU, 'o-', color=col, label=m_name)
    ax[1].plot(sub.bucket, sub['sim(pred, source)'], 'o-', color=col, label=m_name)
ax[0].set_title('BLEU vs input length'); ax[1].set_title('Meaning preservation vs input length')
for a in ax: a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/expC_length.png', dpi=120); plt.show()

ex_long = long_df.source_text[:4].tolist()
display(pd.DataFrame({'long input': ex_long, 'Model A': generate(model_A, ex_long), 'Model B': generate(model_B, ex_long)}))

### Experiment D: Unseen sentences
Hand-written sentences that are not in the dataset, including words the model has never seen
(shown as `<UNK>` count). There is no reference, so we judge meaning preservation with
similarity to the source and inspect the outputs.

In [ ]:
UNSEEN = [
    'My laptop stopped working right before my presentation.',
    'The new coffee shop downtown has terrible service.',
    'I finally finished reading that long novel.',
    'Our football team won the championship last night.',
    'I am nervous about my job interview on Monday.',
    'The weather is absolutely beautiful this morning.',
    'Can you help me move my furniture this weekend?',
    'I think the teacher was unfair to us.',
    'That joke was not funny at all.',
    'I spent all my money on shoes again.',
    'Why is the internet so slow today?',
    'I am proud of how much you have improved.',
    'This assignment is much harder than I expected.',
    'We should order pizza for dinner tonight.',
    'He keeps ignoring my messages.',
]
unk = [sum(t not in SRC_VOCAB.stoi for t in tokenize(s)) for s in UNSEEN]
outA, outB, outBb = generate(model_A, UNSEEN), generate(model_B, UNSEEN), generate(model_B, UNSEEN, 'beam')
unseen_df = pd.DataFrame({'input': UNSEEN, '#UNK': unk, 'Model A': outA, 'Model B greedy': outB, 'Model B beam': outBb,
                          'sim_A': sem_sim(outA, UNSEEN).round(3), 'sim_B': sem_sim(outB, UNSEEN).round(3)})
display(unseen_df)
unseen_df.to_csv(f'{OUT_DIR}/expD_unseen.csv', index=False)
print(f"Mean sim(pred, source)  A: {unseen_df.sim_A.mean():.3f}  B: {unseen_df.sim_B.mean():.3f}  "
      f"| avg <UNK> per sentence: {np.mean(unk):.2f}")

### Experiment E: Meaning preservation, real understanding or slang filler?
Three probes on the test set (Model B, beam):
1. **Matched vs mismatched similarity:** similarity of each output to *its own* source vs to a
   *random other* source. If the model ignored the input and emitted generic slang, both
   numbers would be similar.
2. **Content-word retention:** share of the source's content words (not stopwords, not context
   phrases) that survive in the output.
3. **Slang-only rate:** outputs that contain slang but keep none of the source's content words.

In [ ]:
STOP = set("""a an the i me my we our you your he she it they them this that these those is am are was were be been
being to of in on at for with by from up about into over after before during while and or but so not no do does did
have has had will would can could should very really just too what why how when where who which there here all any
some more most much many such than then now right today tonight""".split())
def content_words(s): return {t for t in tokenize(s) if t.isalpha() and t not in STOP}

hyps = PRED[('B', 'beam')]
matched = sem_sim(hyps, test_src)
shuffled_src = test_src[:]; random.Random(1).shuffle(shuffled_src)
mismatched = sem_sim(hyps, shuffled_src)
retention = [len(content_words(h) & content_words(s)) / max(1, len(content_words(s))) for h, s in zip(hyps, test_src)]
ref_retention = [len(content_words(r) & content_words(s)) / max(1, len(content_words(s))) for r, s in zip(test_ref, test_src)]
slang_only = [bool(slang_terms(h)) and len(content_words(h) & content_words(s)) == 0 for h, s in zip(hyps, test_src)]
expE = pd.DataFrame([
    {'probe': 'sim(output, own source)', 'Model B': matched.mean(), 'Human reference': sem_sim(test_ref, test_src).mean()},
    {'probe': 'sim(output, random source)', 'Model B': mismatched.mean(), 'Human reference': sem_sim(test_ref, shuffled_src).mean()},
    {'probe': 'content-word retention', 'Model B': np.mean(retention), 'Human reference': np.mean(ref_retention)},
    {'probe': 'slang-only outputs (%)', 'Model B': 100 * np.mean(slang_only),
     'Human reference': 100 * np.mean([bool(slang_terms(r)) and not (content_words(r) & content_words(s)) for r, s in zip(test_ref, test_src)])},
]).round(3)
display(expE); expE.to_csv(f'{OUT_DIR}/expE_meaning.csv', index=False)

plt.figure(figsize=(6.5, 3.5))
plt.hist(matched, bins=30, alpha=.7, label='own source', color='#55A868')
plt.hist(mismatched, bins=30, alpha=.7, label='random source', color='#C44E52')
plt.xlabel('cosine similarity'); plt.title('Experiment E: output similarity to source'); plt.legend()
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/expE_meaning.png', dpi=120); plt.show()

worst = pd.DataFrame({'source': test_src, 'output': hyps, 'reference': test_ref, 'sim': matched.round(3)}).sort_values('sim')
print('Lowest meaning preservation (failure cases):'); display(worst.head(8))

## 11. Human evaluation
The assignment requires **at least 3 evaluators** to rate outputs 1–5 on meaning preservation,
grammatical fluency, Gen Z vocabulary appropriateness and naturalness.

1. The cell below writes `human_eval_sheet.csv` (30 random test sentences, Model B beam output)
   and downloads it.
2. Give a copy to each of 3 people. They fill the four rating columns with 1–5.
3. Save the copies as `human_eval_rater1.csv`, `human_eval_rater2.csv`, `human_eval_rater3.csv`.
4. Set `HUMAN_EVAL_READY = True` in the next cell and run it; it uploads the three files and
   computes means, inter-rater agreement, and correlation with BLEU / semantic similarity.

In [ ]:
CRITERIA = ['meaning_preservation', 'grammatical_fluency', 'genz_vocabulary', 'naturalness']
he_idx = test_df.sample(30, random_state=SEED).index
sheet = pd.DataFrame({'item': range(1, 31), 'original_sentence': test_df.loc[he_idx, 'source_text'].values,
                      'genzify_output': [PRED[('B', 'beam')][i] for i in he_idx]})
for c in CRITERIA: sheet[c] = ''
sheet.to_csv(f'{OUT_DIR}/human_eval_sheet.csv', index=False)
display(sheet.head())
files.download(f'{OUT_DIR}/human_eval_sheet.csv')

In [ ]:
HUMAN_EVAL_READY = False   # set True after collecting the 3 filled sheets

if HUMAN_EVAL_READY:
    print('Upload the 3 filled sheets (human_eval_rater1.csv, ...rater2.csv, ...rater3.csv)')
    up = files.upload()
    raters = [pd.read_csv(io.BytesIO(up[k])) for k in sorted(up)]
    assert len(raters) >= 3, 'Need at least 3 evaluators'
    scores = np.stack([r[CRITERIA].astype(float).values for r in raters])          # (R, 30, 4)
    he = pd.DataFrame({'criterion': CRITERIA, 'mean': scores.mean((0, 1)), 'std across raters': scores.std(0).mean(0)})
    # inter-rater agreement: mean pairwise Spearman correlation per criterion
    from itertools import combinations
    from scipy.stats import spearmanr
    he['pairwise Spearman'] = [np.nanmean([spearmanr(scores[a, :, k], scores[b, :, k]).correlation
                                           for a, b in combinations(range(len(raters)), 2)]) for k in range(4)]
    display(he.round(3)); he.to_csv(f'{OUT_DIR}/human_eval_summary.csv', index=False)

    # metric vs human agreement
    item_bleu = [sent_bleu(h, test_df.loc[i, 'target_text']) for h, i in zip(sheet.genzify_output, he_idx)]
    item_sim = sem_sim(sheet.genzify_output.tolist(), sheet.original_sentence.tolist())
    human_meaning = scores[:, :, 0].mean(0); human_nat = scores[:, :, 3].mean(0)
    corr = pd.DataFrame({
        'human meaning': [spearmanr(item_bleu, human_meaning).correlation, spearmanr(item_sim, human_meaning).correlation],
        'human naturalness': [spearmanr(item_bleu, human_nat).correlation, spearmanr(item_sim, human_nat).correlation]},
        index=['sentence BLEU', 'sim(output, source)']).round(3)
    print('Spearman correlation between automatic metrics and human judgments:'); display(corr)
    dis = sheet.assign(BLEU=np.round(item_bleu, 1), sim=item_sim.round(3), human_meaning=human_meaning.round(2))
    dis['gap'] = (dis.human_meaning / 5) - (dis.BLEU / 100)
    print('Largest disagreements (humans like it, BLEU does not):'); display(dis.sort_values('gap', ascending=False).head(5))
else:
    print('Human evaluation pending: fill the sheets, then set HUMAN_EVAL_READY = True and rerun this cell.')

**Expected disagreements between automatic metrics and humans (discuss with your numbers):**
BLEU rewards exact n-gram overlap with one reference, so a correct paraphrase ("I'm so done"
vs "I'm literally dead") scores near zero while humans rate it highly. Conversely, an output that
copies the reference's slang frame but swaps the content noun gets decent BLEU but a poor
meaning score from humans. Embedding similarity tracks meaning better, but the sentence encoder
was trained mostly on standard English and can misread slang ("fire", "dead", "cooked") literally.

## 12. Bonus challenges

### Bonus 3: Reverse transformation (Gen Z → conventional English)
Same architecture and configuration, trained with source and target swapped (own vocabularies).

In [ ]:
REV_SRC_VOCAB, REV_TRG_VOCAB = build_vocabs(train_df, 'target_text', 'source_text')
rev_loaders = make_loaders(FINAL_CFG['batch_size'], REV_SRC_VOCAB, REV_TRG_VOCAB, 'target_text', 'source_text')
model_R, hist_R = train_model(FINAL_CFG, attention=True, epochs=FINAL_EPOCHS, name='Reverse (attention)',
                              loaders=rev_loaders, src_vocab=REV_SRC_VOCAB, trg_vocab=REV_TRG_VOCAB)
plot_history({'Reverse model': hist_R}, 'Bonus 3: reverse model loss', 'loss_reverse.png')

def to_english(sentences, method='beam'):
    return generate(model_R, sentences, method, src_vocab=REV_SRC_VOCAB, trg_vocab=REV_TRG_VOCAB)

rev_hyps = to_english(test_ref)
print(f"Reverse test BLEU: {corpus_bleu(rev_hyps, test_src):.2f} | sim(pred, original English): {sem_sim(rev_hyps, test_src).mean():.3f}")
display(pd.DataFrame({'Gen Z input': test_ref[:8], 'model English': rev_hyps[:8], 'original English': test_src[:8]}))

# round trip: English -> Gen Z -> English
rt = to_english(PRED[('B', 'beam')])
print(f"Round-trip (EN→GenZ→EN) sim to original: {sem_sim(rt, test_src).mean():.3f}")

### Bonus 5: Unknown slang detection
Gen Z slang that never appears in training. A word-level model maps any unseen word to `<UNK>`,
so we (1) **detect** unknown slang by flagging out-of-vocabulary tokens in the reverse model's
input, and (2) **analyse** what the models do with it.

In [ ]:
UNKNOWN_SLANG = ["That song is mid.", "He has so much rizz.", "She is being delulu again.",
                 "This party is giving skibidi energy.", "Bro is yapping too much.",
                 "That fit is drip fr.", "I'm feeling salty about the exam.", "You need to touch grass bro.",
                 "That was a total ick.", "The plot twist was bussin no cap."]

def detect_unknown_slang(text, vocab=None):
    vocab = vocab or REV_SRC_VOCAB
    return [t for t in tokenize(text) if t.isalpha() and t not in vocab.stoi]

slang_rows = []
for s in UNKNOWN_SLANG:
    oov = detect_unknown_slang(s)
    slang_rows.append({'Gen Z input': s, 'unknown tokens': ', '.join(oov) or '(none)',
                       'reverse → English': to_english([s])[0],
                       'forward GenZify(input)': generate(model_B, [s], 'beam')[0]})
slang_df = pd.DataFrame(slang_rows); display(slang_df)
slang_df.to_csv(f'{OUT_DIR}/bonus5_unknown_slang.csv', index=False)
print('Detection rate of unknown slang words:',
      f"{np.mean([bool(detect_unknown_slang(s)) for s in UNKNOWN_SLANG]):.0%} of inputs flagged")

**Analysis:** every unseen slang word becomes `<UNK>`, so the model cannot know whether "mid"
is positive or negative; it falls back on the surrounding words and on frequent training
patterns. The detector reliably flags the new terms (useful for a "learn this word" loop), but
translating them would need either more data, sub-word tokenisation (BPE) so pieces like
"delulu" ← "delusional" can be shared, or a pre-trained language model.

### Bonus 4: Context-aware slang
Does the model choose slang that fits the topic? We build a per-category slang profile from the
training targets and check whether slang in test outputs matches the *category's* profile more
often than chance (the overall slang distribution).

In [ ]:
if 'category' in df.columns:
    cat_slang = {c: Counter(s for t in train_df[train_df.category == c].target_text for s in slang_terms(t)) for c in train_df.category.unique()}
    global_slang = Counter(s for t in train_df.target_text for s in slang_terms(t))
    hits, base, n = 0, 0.0, 0
    for h, c in zip(PRED[('B', 'beam')], test_df.category):
        for s in slang_terms(h):
            n += 1
            cat_p = cat_slang[c][s] / max(1, sum(cat_slang[c].values()))
            glob_p = global_slang[s] / sum(global_slang.values())
            hits += cat_p > glob_p; base += 0.5
    print(f'Slang tokens in test outputs: {n}; share that are over-represented in their own category: {hits / max(1, n):.1%}')
    profile = pd.DataFrame({c: [', '.join(s for s, _ in cat_slang[c].most_common(4))] for c in cat_slang}).T
    profile.columns = ['top slang in training targets']; display(profile)

CONTEXT_PROBES = ['This restaurant has excellent food.', 'I am very confident about my exam.',
                  'I am exhausted after the meeting.', 'That outfit looks very stylish.',
                  'I cannot believe you did that.', 'We made the right decision.']
display(pd.DataFrame({'input': CONTEXT_PROBES, 'GenZify': generate(model_B, CONTEXT_PROBES, 'beam')}))

The attention heatmaps in Experiment A are the second piece of evidence: slang tokens such as
"bussin" or "locked in" attend to the source words that trigger them ("food", "confident"),
i.e. the slang choice is conditioned on the content rather than inserted at random.

### Bonus 2: Multiple outputs and Bonus 1: Tone selector
**Multiple outputs:** the n-best list from beam search plus temperature/top-k sampling,
de-duplicated, gives three different Gen Z versions.

**Tone selector:** the dataset only has two tones (casual, reactive), so a neural model cannot
learn "funny" or "sarcastic" from it. We implement tone as a **hybrid**: the neural model
produces candidates, then a tone layer (a) re-ranks candidates using tone-specific slang
lexicons and (b) applies light tone-specific framing. This is stated honestly as a
rule-assisted feature; a fully learned version would need tone-labelled training pairs and a
tone control token prepended to the source.

In [ ]:
@torch.no_grad()
def sample_decode(model, sentence, temperature=0.9, top_k=8):
    model.eval()
    ids = SRC_VOCAB.encode(tokenize(sentence))
    src = torch.tensor([ids], device=DEVICE); lens = torch.tensor([len(ids)])
    enc_out, h, c, mask, enc_proj = model.encode(src, lens)
    tok = torch.tensor([SOS_IDX], device=DEVICE); out = []
    for _ in range(max_len_for(len(ids))):
        logits, h, c, att = model.dec_step(tok, h, c, enc_out, mask, enc_proj)
        logits = logits / temperature
        v, i = logits.topk(top_k)
        tok = i[0, torch.multinomial(F.softmax(v[0], -1), 1)]
        if tok.item() == EOS_IDX: break
        if tok.item() == UNK_IDX:
            if att is not None: out.append(tokenize(sentence)[min(len(ids) - 3, int(att[0, 1:-1].argmax()))])
            continue
        out.append(TRG_VOCAB.itos[tok.item()])
    return detokenize(out)

def multiple_outputs(sentence, n=3, model=None):
    model = model or model_B
    cands = [detokenize(t) for t, _ in beam_search(model, sentence, beam_width=max(5, n * 2), n_best=n * 2)]
    tries = 0
    while len(dict.fromkeys(cands)) < n and tries < 20:
        cands.append(sample_decode(model, sentence)); tries += 1
    uniq = list(dict.fromkeys(cands))
    return uniq[:n]

TONES = {
    'Casual Gen Z':    dict(words=['lowkey', 'fr', 'rn', 'vibe', 'vibing', 'no cap'], prefix='', suffix=''),
    'Funny Gen Z':     dict(words=['dead', 'cooked', 'ain\'t no way', 'bro', 'crying'], prefix='', suffix=' 💀😭'),
    'Sarcastic Gen Z': dict(words=['sure', 'totally', 'love that', 'great'], prefix='Oh wow, ', suffix=' Love that for us 🙄'),
    'Friendly Gen Z':  dict(words=['bestie', 'fr', 'love', 'no cap', 'w'], prefix='Bestie, ', suffix=' 🫶'),
}

def apply_tone(sentence, tone='Casual Gen Z', n_candidates=6):
    spec = TONES[tone]
    cands = [detokenize(t) for t, _ in beam_search(model_B, sentence, beam_width=n_candidates, n_best=n_candidates)]
    def tone_score(c): return sum(w in c.lower() for w in spec['words'])
    best = max(cands, key=lambda c: (tone_score(c), -cands.index(c)))   # rerank; ties keep beam order
    if spec['prefix'] and not best.lower().startswith(spec['prefix'].lower().strip(', ')):
        rest = best if re.match(r"I\b|I'", best) else best[0].lower() + best[1:]   # keep "I" capitalised
        best = spec['prefix'] + rest
    return best.rstrip() + spec['suffix']

s = 'I really enjoyed spending time with my friends.'
print('Input:', s)
for i, alt in enumerate(multiple_outputs(s), 1): print(f'  Alternative {i}: {alt}')
for t in TONES: print(f'  {t:16s}: {apply_tone(s, t)}')

## 13. GenZify application (Gradio)
Features: input textbox, **Convert** button, generated output, display of the original
sentence, model + decoding choice, tone selector (Bonus 1), multiple alternatives (Bonus 2),
a reverse-translation tab (Bonus 3) with unknown-slang detection (Bonus 5), and **Clear**.
Run the cell, open the public link, and take screenshots for the deliverables.

In [ ]:
import gradio as gr

def genzify_app(text, model_choice, decoding, tone, n_alts):
    text = (text or '').strip()
    if not text:
        return '', 'Please type a sentence.', ''
    model = model_B if model_choice.startswith('B') else model_A
    if tone != 'None (model only)':
        out = apply_tone(text, tone)
    else:
        out = generate(model, [text], 'beam' if decoding.startswith('Beam') else 'greedy')[0]
    alts = ''
    if n_alts > 1:
        alts = '\n'.join(f'{i}. {a}' for i, a in enumerate(multiple_outputs(text, int(n_alts), model), 1))
    unk = [t for t in tokenize(text) if t.isalpha() and t not in SRC_VOCAB.stoi]
    note = f'  (words not in training vocabulary: {", ".join(unk)})' if unk else ''
    return f'Original: {text}{note}', out, alts

def reverse_app(text):
    text = (text or '').strip()
    if not text: return '', ''
    oov = detect_unknown_slang(text)
    return to_english([text])[0], ('Unknown slang detected: ' + ', '.join(oov)) if oov else 'No unknown slang detected.'

with gr.Blocks(title='GenZify') as demo:
    gr.Markdown('# 🔥 GenZify\nTurn conventional (Millennial) English into Gen Z style with an LSTM encoder–decoder.')
    with gr.Tab('English → Gen Z'):
        inp = gr.Textbox(label='Conventional English', placeholder='That is an excellent idea!', lines=2)
        with gr.Row():
            model_choice = gr.Radio(['B: Attention', 'A: Basic'], value='B: Attention', label='Model')
            decoding = gr.Radio(['Beam search (k=5)', 'Greedy'], value='Beam search (k=5)', label='Decoding')
        with gr.Row():
            tone = gr.Dropdown(['None (model only)'] + list(TONES), value='None (model only)', label='Tone (bonus)')
            n_alts = gr.Slider(1, 3, value=1, step=1, label='Number of alternatives')
        with gr.Row():
            convert = gr.Button('Convert ✨', variant='primary'); clear = gr.Button('Clear')
        orig = gr.Textbox(label='Original sentence', interactive=False)
        out = gr.Textbox(label='Gen Z output', interactive=False)
        alts = gr.Textbox(label='Alternatives', interactive=False, lines=3)
        convert.click(genzify_app, [inp, model_choice, decoding, tone, n_alts], [orig, out, alts])
        clear.click(lambda: ('', '', '', '', 'None (model only)', 1), None, [inp, orig, out, alts, tone, n_alts])
        gr.Examples(DEMO[:6], inp)
    with gr.Tab('Gen Z → English (reverse)'):
        rin = gr.Textbox(label='Gen Z text', placeholder="That's actually fire, no cap.", lines=2)
        with gr.Row():
            rbtn = gr.Button('Translate back', variant='primary'); rclear = gr.Button('Clear')
        rout = gr.Textbox(label='Conventional English', interactive=False)
        rnote = gr.Textbox(label='Unknown slang check', interactive=False)
        rbtn.click(reverse_app, rin, [rout, rnote])
        rclear.click(lambda: ('', '', ''), None, [rin, rout, rnote])

demo.launch(share=True, debug=False)

## 14. Save models, results and the comparison report

In [ ]:
def save_checkpoint(model, cfg, sv, tv, path, attention):
    torch.save({'state_dict': model.state_dict(), 'config': cfg, 'attention': attention,
                'src_itos': sv.itos, 'trg_itos': tv.itos}, path)

save_checkpoint(model_A, FINAL_CFG, SRC_VOCAB, TRG_VOCAB, f'{OUT_DIR}/genzify_model_A_basic.pt', False)
save_checkpoint(model_B, FINAL_CFG, SRC_VOCAB, TRG_VOCAB, f'{OUT_DIR}/genzify_model_B_attention.pt', True)
save_checkpoint(model_R, FINAL_CFG, REV_SRC_VOCAB, REV_TRG_VOCAB, f'{OUT_DIR}/genzify_reverse_model.pt', True)
pd.DataFrame({'source': test_src, 'reference': test_ref, **{f'{m}_{d}': v for (m, d), v in PRED.items()}}) \
  .to_csv(f'{OUT_DIR}/test_predictions.csv', index=False)
df[['source_text', 'target_text'] + [c for c in ['category', 'tone'] if c in df.columns] + ['split']] \
  .to_csv(f'{OUT_DIR}/dataset_used.csv', index=False)
json.dump({'A': hist_A, 'B': hist_B, 'reverse': hist_R}, open(f'{OUT_DIR}/loss_history.json', 'w'))

In [ ]:
# Auto-generated comparison report built from this run's numbers
r = results_df.set_index(['model', 'decoding'])
bA, bB = r.loc[('A: basic', 'greedy'), 'BLEU'], r.loc[('B: attention', 'greedy'), 'BLEU']
bBb = r.loc[('B: attention', 'beam'), 'BLEU']
sA, sB = r.loc[('A: basic', 'greedy'), 'sim(pred, source)'], r.loc[('B: attention', 'greedy'), 'sim(pred, source)']
sRef = r.loc[('Human reference', '-'), 'sim(pred, source)']
lc = len_df.set_index(['bucket', 'model'])
drop = lambda m: lc.loc[('short (5-8)', m), 'BLEU'] - lc.loc[('long (20-30)', m), 'BLEU']

report = f"""# GenZify: Comparison Report

## Setup
* Dataset: {len(df)} parallel pairs (train {len(train_df)} / validation {len(val_df)} / test {len(test_df)}); no base sentence is shared across splits.
* Vocabulary: {len(SRC_VOCAB)} source / {len(TRG_VOCAB)} target tokens (+ <PAD>, <SOS>, <EOS>, <UNK>).
* Final configuration: {FINAL_CFG}
* Encoder: bi-directional {FINAL_CFG['n_layers']}-layer LSTM; Model A decoder uses only the final (h, c); Model B adds Bahdanau attention.

## Test results (unseen base sentences)
{results_df.to_markdown(index=False)}

## Experiment A: basic vs attention
Greedy BLEU {bA:.2f} (A) vs {bB:.2f} (B); meaning preservation sim(pred, source) {sA:.3f} vs {sB:.3f}.
Attention {'improved' if bB > bA else 'did not improve'} BLEU by {bB - bA:+.2f} points. Best validation BLEU: A {hist_A['best_val_bleu']:.2f}, B {hist_B['best_val_bleu']:.2f}.

## Experiment B: greedy vs beam
{beam_df.to_markdown(index=False)}

## Experiment C: sentence length
{len_df.to_markdown(index=False)}

BLEU drop from short to long inputs: A {drop('A: basic'):.2f}, B {drop('B: attention'):.2f} points.
{'Attention degrades less on long inputs, consistent with the fixed context vector being an information bottleneck.' if drop('B: attention') < drop('A: basic') else 'Both models degrade on long inputs; long inputs exceed every training length.'}

## Experiment D: unseen sentences
Mean sim(pred, source): A {unseen_df.sim_A.mean():.3f}, B {unseen_df.sim_B.mean():.3f}; average {np.mean(unk):.2f} <UNK> tokens per sentence.

## Experiment E: meaning preservation
{expE.to_markdown(index=False)}

## Final research question
*Can a neural encoder-decoder learn conversational style transformation while preserving meaning?*
On sentences whose content it has never seen, the attention model reaches BLEU {bBb:.2f} (beam) and an
output-to-source similarity of {r.loc[('B: attention', 'beam'), 'sim(pred, source)']:.3f} against {sRef:.3f} for the human-written references.
Its outputs are {expE.iloc[0, 1] - expE.iloc[1, 1]:.3f} more similar to their own source than to a random one,
so it conditions on the input rather than emitting generic slang. Style transfer (slang, contractions, "rn",
"fr") is learned reliably; meaning preservation is partial and breaks down for words outside the
{len(SRC_VOCAB)}-word vocabulary and for inputs longer than those seen in training.
"""
open(f'{OUT_DIR}/comparison_report.md', 'w').write(report)
print(report)

In [ ]:
with zipfile.ZipFile('genzify_deliverables.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(OUT_DIR):
        z.write(os.path.join(OUT_DIR, f), f)
print('Contents:', sorted(os.listdir(OUT_DIR)))
files.download('genzify_deliverables.zip')

## Deliverables checklist
1. **Dataset (CSV):** `dataset_used.csv` ✔
2. **Python implementation:** this notebook ✔
3. **Trained encoder–decoder model:** `genzify_model_A_basic.pt` ✔
4. **Attention model:** `genzify_model_B_attention.pt` (+ reverse model) ✔
5. **Training/validation loss graphs:** `loss_*.png`, `hyperparameter_sweep_curves.png` ✔
6. **Evaluation results:** `test_results.csv`, `exp*_*.csv`, `test_predictions.csv` ✔
7. **Screenshots of the app:** take them from the Gradio link in §13
8. **Comparison report:** `comparison_report.md` ✔
9. **Short presentation:** use §4's architecture description, the loss plots, the attention
   heatmaps and the report's tables as slide content.